In [ ]:
# from google.colab import runtime
# runtime.unassign()

In [1]:
!rm -rf sample_data/

In [2]:
# dependencies imports and checks

import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np
try:
  import mlflow
except:
  %pip install mlflow

try:
  import torchvision
except:
  %pip install torchvision

import torchvision
from torchvision import datasets
from torchvision import models
from torch.utils.data import DataLoader
from torchvision import transforms
from pathlib import Path
print(f"Pytorch version: {torch.__version__}\ntorchvision version:{torchvision.__version__}")

import glob

try:
  import torchinfo
except:
   %pip install torchinfo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!pwd

/content


In [4]:
!ls

data_setup.py  engine.py  model.py  train.ipynb  utils.py


In [5]:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# import os
# os.chdir("/content/drive/MyDrive/VLM_pipeline")
# print(os.getcwd())

In [6]:
# Modular imports
from data_setup import create_datasets, get_data
from engine import train
from model import ViT
from utils import (save_model, load_model, display_random_images_from_dataset,
                   set_seed, pred_and_plot_image, plot_loss_acc_curves,
                   plot_random_images_from_path, get_device,
                   walk_through_dir,
                   plot_confusion_matrix,
                   print_patched_image)

In [7]:
from torch.profiler import profile, ProfilerActivity, record_function

In [8]:
device = get_device()
device

'cuda'

In [22]:
!rm -rf /content/data/

In [23]:
get_data()

Did not find /content/data/pizza_steak_sushi directory, creating one...
Unzipping pizza, steak, sushi data...


In [24]:
train_dir = Path('./data/pizza_steak_sushi/train')
test_dir = Path('./data/pizza_steak_sushi/test')

In [25]:
train_images_paths = list((train_dir).glob('*/*.jpg'))
train_images_paths = train_images_paths[:-32]
test_images_paths = list((test_dir).glob('*/*.jpg'))
test_images_paths = test_images_paths[:-32]

In [26]:
import os
for train_image in train_images_paths:
    os.remove(train_image)

for test_image in test_images_paths:
    os.remove(test_image)

In [27]:
walk_through_dir(train_dir)

there are 3 directories and 0 images in inside data/pizza_steak_sushi/train directory
there are 0 directories and 0 images in inside data/pizza_steak_sushi/train/steak directory
there are 0 directories and 0 images in inside data/pizza_steak_sushi/train/pizza directory
there are 0 directories and 32 images in inside data/pizza_steak_sushi/train/sushi directory


In [28]:
walk_through_dir(test_dir)

there are 3 directories and 0 images in inside data/pizza_steak_sushi/test directory
there are 0 directories and 0 images in inside data/pizza_steak_sushi/test/steak directory
there are 0 directories and 1 images in inside data/pizza_steak_sushi/test/pizza directory
there are 0 directories and 31 images in inside data/pizza_steak_sushi/test/sushi directory


In [29]:
training_resolution = 224

vit_preprocess_transformations = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((training_resolution,training_resolution))
])

In [30]:
vit_train_dataloader, vit_test_dataloader, vit_class_names, vit_class_to_idx = create_datasets(
                    train_dir = train_dir,
                    test_dir = test_dir,
                    train_transformations = vit_preprocess_transformations,
                    test_transformations= vit_preprocess_transformations,
                    NUM_WORKERS= 0,
                    BATCH_SIZE = 32,
                    PIN_MEMORY = False,
                    DROP_LAST= True)

In [31]:
set_seed(42)
# reduce regularization for now and try to avoid underfitting over training data
ViT_model = ViT(patch_projection_size=128,
               patch_resolution=16,
               num_patches=196,
               in_channels=3,
               num_transformer_layers=2,
               dropout_rate=0,
               num_heads=4,
               MLP_size=256,
               num_classes = len(vit_class_names))

In [32]:
print("Dataset size:", len(vit_train_dataloader.dataset))
print("Number of batches:", len(vit_train_dataloader))
print("Batch size:", vit_train_dataloader.batch_size)


Dataset size: 32
Number of batches: 1
Batch size: 32


In [33]:
print("Train samples:", len(vit_train_dataloader.dataset))
print("Train batches:", len(vit_train_dataloader))
print("Test samples:", len(vit_test_dataloader.dataset))
print("Test batches:", len(vit_test_dataloader))


Train samples: 32
Train batches: 1
Test samples: 32
Test batches: 1


In [34]:
# Vanilla ViT
from engine import train
from utils import get_device

device = get_device()
NUM_EPOCHS = 500

vit_optimizer = torch.optim.Adam(params = ViT_model.parameters(),
                                 lr=0.001,
                                  betas=(0.9,0.999))

vit_loss_fn = torch.nn.CrossEntropyLoss()

vit_train_results = train(model = ViT_model,
                train_dataloader = vit_train_dataloader,
                test_dataloader = vit_test_dataloader,
                optimizer = vit_optimizer,
                device = device,
                loss_fn = vit_loss_fn,
                epochs = NUM_EPOCHS
                )

  0%|          | 0/500 [00:00<?, ?it/s]

Completed 1 batch out of 1 for training
Completed 1 batch out of 1 for testing
Epoch: 1 | train_loss: 1.5302 | train_acc: 0.0000 | test_loss: 0.1567 | test_acc: 0.9688
Completed 1 batch out of 1 for training
Completed 1 batch out of 1 for testing
Epoch: 2 | train_loss: 0.0229 | train_acc: 1.0000 | test_loss: 0.1626 | test_acc: 0.9688
Completed 1 batch out of 1 for training
Completed 1 batch out of 1 for testing
Epoch: 3 | train_loss: 0.0149 | train_acc: 1.0000 | test_loss: 0.1721 | test_acc: 0.9688
Completed 1 batch out of 1 for training
Completed 1 batch out of 1 for testing
Epoch: 4 | train_loss: 0.0096 | train_acc: 1.0000 | test_loss: 0.1830 | test_acc: 0.9688
Completed 1 batch out of 1 for training
Completed 1 batch out of 1 for testing
Epoch: 5 | train_loss: 0.0063 | train_acc: 1.0000 | test_loss: 0.1938 | test_acc: 0.9688
Completed 1 batch out of 1 for training
Completed 1 batch out of 1 for testing
Epoch: 6 | train_loss: 0.0043 | train_acc: 1.0000 | test_loss: 0.2036 | test_acc:

KeyboardInterrupt: 

In [ ]:
plot_loss_acc_curves(vit_train_results)


In [ ]:
from utils import plot_confusion_matrix
plot_confusion_matrix(ViT_model,vit_test_dataloader,'cuda',vit_class_names)

In [ ]:
# from datetime import datetime
# model_name = '_epochs_'+ datetime.now().strftime('%Y_Y_%m_m_%d_d_%H:%M:%S')+'_VGG16MINI.pth'
# save_model(model=model,
#            target_dir='/content/models',
#            model_name=model_name)


In [ ]:
# !rm -rf /content/drive/MyDrive/VLM_pipeline/data/pizza_steak_sushi

In [ ]:
from torchvision.datasets import FashionMNIST

train_fashion_mnist = FashionMNIST()